#  Part 1:  Recurrent Neural Network

In [3]:
!pip install numpy
import numpy as np
#Restart the kernel after running this code

###  Importing packages

In [4]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.datasets import imdb

from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings('ignore')
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

### Preparing Dataset

Loads the IMDB movie reviews dataset - 50,000 reviews labeled as positive or negative sentiment

Keeps only the top 1,000 most frequent words - Reviews are now represented as sequences of numbers (word IDs from 1-1000)

Makes all reviews the same length (80 words):

Short reviews (<80 words) → Add zeros at the beginning

Long reviews (>80 words) → Cut off words after position 80

x_train[0] = [12, 45, 3, 789, 23, 1, 456]  (review text as numbers)
y_train[0] = 1                               (positive sentiment)

x_test[0] = [67, 234, 8, 99, 4]            (different review)
y_test[0] = 0                               (negative sentiment)

In [5]:

max_features = 1000
maxlen = 80  # cut texts after this number of words (among top max_features most common words)
batch_size = 32

print('Loading data...')
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
print(len(x_train), 'train sequences')
print(len(x_test), 'test sequences')

print('Pad sequences (samples x time)')
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)
print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)

Loading data...
25000 train sequences
25000 test sequences
Pad sequences (samples x time)
x_train shape: (25000, 80)
x_test shape: (25000, 80)


In [6]:
y_train[0]

np.int64(1)

### Visualize the data

In [7]:
INDEX_FROM=3   # word index offset

word_to_id = imdb.get_word_index()
word_to_id = {k:(v+INDEX_FROM) for k,v in word_to_id.items()}
word_to_id["<PAD>"] = 0
word_to_id["<START>"] = 1
word_to_id["<UNK>"] = 2

id_to_word = {value:key for key,value in word_to_id.items()}
print(' '.join(id_to_word[id] for id in x_train[0] ))

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
that played the <UNK> of <UNK> and paul they were just brilliant children are often left out of the <UNK> <UNK> i think because the stars that play them all <UNK> up are such a big <UNK> for the whole film but these children are amazing and should be <UNK> for what they have done don't you think the whole story was so <UNK> because it was true and was <UNK> life after all that was <UNK> with us all


Simple formula for LSTM Parameters -  4 × (units × (units + input_dim) + units)

### Building a Model

In [8]:
model = Sequential()
model.add(Embedding(max_features, 8))
model.add(LSTM(16, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))
model.build(input_shape=(None, maxlen))  # ← Build with input shape
model.summary()  # Now shows real numbers!



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 80, 8)          │         8,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 16)             │         1,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,617 (37.57 KB)

 Trainable params: 9,617 (37.57 KB)

 Non-trainable params: 0 (0.00 B)

### Model Training

In [9]:
# try using different optimizers and different optimizer configs
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Write the training input and output, batch size, and testing input and output

model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=1,
          validation_data=(x_test, y_test))

782/782 ━━━━━━━━━━━━━━━━━━━━ 47s 56ms/step - accuracy: 0.6952 - loss: 0.5789 - val_accuracy: 0.7794 - val_loss: 0.4788


### Testing

In [10]:
score, acc = model.evaluate(x_test, y_test, batch_size=batch_size)
print('Test score:', score)
print('Test accuracy:', acc)

782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.7794 - loss: 0.4788
Test score: 0.47876355051994324
Test accuracy: 0.7794399857521057


### Prediction

In [11]:
# ADD THIS LINE - Shows the review
print('Review:', ' '.join([id_to_word[id] for id in x_test[2] if id > 2]))

# YOUR ORIGINAL CODE
prediction = model.predict(x_test[2:3])
print('Prediction value:', prediction[0])
print('Test Label:', y_test[2:3])

Review: events may or may not have had in mind when he made but whatever his of material the film as a tale of could be the or in the or any country of any era that its down and is by it's a film even a one in its way but its message is no joke
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step
Prediction value: [0.44130626]
Test Label: [1]


### Other RNN Layers

* keras.layers.RNN(cell, return_sequences=False)
* keras.layers.SimpleRNN(units, activation='tanh')
* keras.layers.GRU(units, activation='tanh', recurrent_activation='hard_sigmoid')
* keras.layers.ConvLSTM2D(filters, kernel_size, strides=(1, 1), padding='valid', )
* keras.layers.SimpleRNNCell(units, activation='tanh')
* keras.layers.GRUCell(units, activation='tanh', recurrent_activation='hard_sigmoid')
* keras.layers.LSTMCell(units, activation='tanh', recurrent_activation='hard_sigmoid')
* keras.layers.CuDNNGRU(units, kernel_initializer='glorot_uniform')
* keras.layers.CuDNNLSTM(units, kernel_initializer='glorot_uniform')

# Part 2: Recurrent Neural Network with Custom Dataset

In [12]:
# Credits to Peter Nagy

In [13]:
import urllib.request
import zipfile
import os

# Downloading a sample sentiment dataset (Sentiment140) to act as Senti.csv
url = 'http://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip'
urllib.request.urlretrieve(url, 'data.zip')

with zipfile.ZipFile('data.zip', 'r') as zip_ref:
    zip_ref.extractall('data_dir')

# Use the training file and rename it
df_temp = pd.read_csv('data_dir/training.1600000.processed.noemoticon.csv',
                     encoding='latin-1', header=None)
df_temp = df_temp[[5, 0]]
df_temp.columns = ['text', 'sentiment']
df_temp['sentiment'] = df_temp['sentiment'].replace({0: 'Negative', 4: 'Positive'})

# Save as Senti.csv so your existing logic works
df_temp.sample(2000).to_csv('Senti.csv', index=False)
print('Senti.csv has been created.')

Senti.csv has been created.


### Load data

In [14]:
import pandas as pd

# Now that Senti.csv is created, we load it
data = pd.read_csv('Senti.csv')
# Keeping only the necessary columns
data = data[['text','sentiment']]
print('Data loaded successfully. Shape:', data.shape)
display(data.head())

Data loaded successfully. Shape: (2000, 2)


,text,sentiment
0,The fish are not biting today! Alice,Negative
1,I feel like poop. I hate being sick,Negative
2,Just put a ton of Mandy Moore on my iPod. It's...,Negative
3,@filiber Hi! Should fine all the info here: ht...,Negative
4,@trevorp no problem. I like learning about wha...,Positive


### Visualize data

In [15]:
data.head(10)

,text,sentiment
0,The fish are not biting today! Alice,Negative
1,I feel like poop. I hate being sick,Negative
2,Just put a ton of Mandy Moore on my iPod. It's...,Negative
3,@filiber Hi! Should fine all the info here: ht...,Negative
4,@trevorp no problem. I like learning about wha...,Positive
5,The rain is so freaking sexy! I cant explain m...,Positive
6,@SoullaStylianou its all in the thermal mass o...,Positive
7,Another busy day at ESEM today...some exciting...,Positive
8,@korennn i hear koren has been a bit silly,Positive
9,"@MrPoofyPJPants I always get headaches, never ...",Negative


### Format data

In [16]:
data = data[data.sentiment != "Neutral"]
data['text'] = data['text'].apply(lambda x: x.lower())
data['text'] = data['text'].apply((lambda x: re.sub('[^a-zA-z0-9\s]','',x)))

for idx,row in data.iterrows():
    data.at[idx, 'text'] = row['text'].replace('rt',' ')

max_fatures = 2000
# Updated nb_words to num_words for modern Keras compatibility
tokenizer = Tokenizer(num_words=max_fatures, split=' ')
tokenizer.fit_on_texts(data['text'].values)
X = tokenizer.texts_to_sequences(data['text'].values)
X = pad_sequences(X)
display(X.shape)

(2000, 32)

In [17]:
data = data[data.sentiment != "Neutral"]
data['text'] = data['text'].apply(lambda x: x.lower())
data['text'] = data['text'].apply((lambda x: re.sub('[^a-zA-z0-9\s]','',x)))

for idx,row in data.iterrows():
    row[0] = row[0].replace('rt',' ')

max_fatures = 2000
tokenizer = Tokenizer(nb_words=max_fatures, split=' ')
tokenizer.fit_on_texts(data['text'].values)
X = tokenizer.texts_to_sequences(data['text'].values)
X = pad_sequences(X)

### Training set

In [18]:
Y = pd.get_dummies(data['sentiment']).values
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.33, random_state = 42)
print('Shape of training samples:',X_train.shape,Y_train.shape)
print('Shape of testing samples:',X_test.shape,Y_test.shape)

Shape of training samples: (1340, 32) (1340, 2)
Shape of testing samples: (660, 32) (660, 2)


### Design a model

In [20]:
from tensorflow.keras.layers import SpatialDropout1D

model = Sequential()
model.add(Embedding(max_fatures, 128))
model.add(SpatialDropout1D(0.2))
model.add(LSTM(128))
model.add(Dense(2, activation='softmax'))
model.compile(loss = 'categorical_crossentropy', optimizer='adam',metrics = ['accuracy'])
print(model.summary())

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


### Training

In [21]:
batch_size = 32
model.fit(X_train, Y_train, epochs = 5, batch_size=batch_size, verbose = 2)

Epoch 1/5
42/42 - 4s - 98ms/step - accuracy: 0.5604 - loss: 0.6868
Epoch 2/5
42/42 - 3s - 81ms/step - accuracy: 0.7269 - loss: 0.5601
Epoch 3/5
42/42 - 2s - 41ms/step - accuracy: 0.8284 - loss: 0.3768
Epoch 4/5
42/42 - 2s - 59ms/step - accuracy: 0.9127 - loss: 0.2407
Epoch 5/5
42/42 - 2s - 40ms/step - accuracy: 0.9410 - loss: 0.1613


### Validation

In [22]:
score,acc = model.evaluate(X_test, Y_test, verbose = 2, batch_size = batch_size)
print("Score: %.2f" % (score))
print("Accuracy: %.2f" % (acc))

21/21 - 1s - 53ms/step - accuracy: 0.6227 - loss: 1.0312
Score: 1.03
Accuracy: 0.62


### Formatting Test Example

In [23]:
text = 'I like him'
tester = np.array([text])
tester = pd.DataFrame(tester)
tester.columns = ['text']

tester['text'] = tester['text'].apply(lambda x: x.lower())
tester['text'] = tester['text'].apply((lambda x: re.sub('[^a-zA-z0-9\s]','',x)))

max_fatures = 2000
test = tokenizer.texts_to_sequences(tester['text'].values)
test = pad_sequences(test)

if X.shape[1]>test.shape[1]:
    test = np.pad(test[0], (X.shape[1]-test.shape[1],0), 'constant')

test = np.array([test])

prediction = model.predict(test)
print('Prediction value:',prediction[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
Prediction value: [0.23002821 0.7699718 ]


# Part 3: RNN Design Choices

## Influence of number of nodes

### LSTM with 8 nodes

Simple formula for LSTM Parameters -  4 × (units × (units + input_dim) + units)

In [24]:
model = Sequential()
model.add(Embedding(max_features, 8))
model.add(LSTM(8, dropout=0.0, recurrent_dropout=0.0))
model.add(Dense(1, activation='sigmoid'))
model.summary()

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_train, y_train, batch_size=batch_size, epochs=1, validation_data=(x_test, y_test))

score, acc = model.evaluate(x_test, y_test, batch_size=batch_size)
print('Test score:', score)
print('Test accuracy:', acc)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 23ms/step - accuracy: 0.7352 - loss: 0.5208 - val_accuracy: 0.8123 - val_loss: 0.4141
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8123 - loss: 0.4141
Test score: 0.4141046106815338
Test accuracy: 0.8123199939727783


### LSTM with 16 nodes

In [25]:
# Write your code here

# Use the same layer design from the above cell

## Influence of Embedding

In [26]:
model = Sequential()
model.add(Embedding(max_features, 4))
model.add(LSTM(16, dropout=0.0, recurrent_dropout=0.0))
model.add(Dense(1, activation='sigmoid'))
model.summary()

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_train, y_train, batch_size=batch_size, epochs=1, validation_data=(x_test, y_test))

score, acc = model.evaluate(x_test, y_test, batch_size=batch_size)
print('Test score:', score)
print('Test accuracy:', acc)

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

782/782 ━━━━━━━━━━━━━━━━━━━━ 22s 25ms/step - accuracy: 0.7289 - loss: 0.5213 - val_accuracy: 0.8128 - val_loss: 0.4090
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8128 - loss: 0.4090
Test score: 0.40899211168289185
Test accuracy: 0.8127999901771545


## Influence of Dropout

### Dropout with probability 0.5

In [27]:
model = Sequential()
model.add(Embedding(max_features, 32))
model.add(LSTM(8, dropout=0.5, recurrent_dropout=0.5))
model.add(Dense(1, activation='sigmoid'))
model.summary()

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_train, y_train, batch_size=batch_size, epochs=1, validation_data=(x_test, y_test))

score, acc = model.evaluate(x_test, y_test, batch_size=batch_size)
print('Test score:', score)
print('Test accuracy:', acc)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

782/782 ━━━━━━━━━━━━━━━━━━━━ 45s 53ms/step - accuracy: 0.6810 - loss: 0.5981 - val_accuracy: 0.7392 - val_loss: 0.5270
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.7392 - loss: 0.5270
Test score: 0.526955783367157
Test accuracy: 0.7391999959945679


### Dropout with probability 0.9

In [ ]:
# Write your code here

# Use the same model design from the above cell

## Multilayered RNNs

### RNN with 2 layer LSTM

In [28]:
model = Sequential()
model.add(Embedding(max_features, 8))
model.add(LSTM(8, dropout=0.0, recurrent_dropout=0.0, return_sequences=True))
model.add(LSTM(8, dropout=0.0, recurrent_dropout=0.0))
model.add(Dense(1, activation='sigmoid'))
model.summary()

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_train, y_train, batch_size=batch_size, epochs=1, validation_data=(x_test, y_test))

score, acc = model.evaluate(x_test, y_test, batch_size=batch_size)
print('Test score:', score)
print('Test accuracy:', acc)

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

782/782 ━━━━━━━━━━━━━━━━━━━━ 37s 42ms/step - accuracy: 0.7363 - loss: 0.5122 - val_accuracy: 0.8048 - val_loss: 0.4226
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8048 - loss: 0.4226
Test score: 0.4226367473602295
Test accuracy: 0.8047999739646912


### RNN with 3 layer LSTM

In [ ]:
# Write your code here

# Use the same node design from the above cell

### What are your findings?